# ferrosim reference corpus — tour

The DB registers **imported third-party circuits** in the same `circuits/` registry as its
verifiable circuits, marked **`kind: reference`** (plan D-9). The **ferrosim** corpus
([`Arcadia-1/ferrosim`](https://github.com/Arcadia-1/ferrosim), author **Token Zhang**, MIT)
is 22 such circuits (92 Spectre decks) in a proprietary 28/65 nm PDK.

They are **not lowered to an open PDK and not simulated here** — the harness runs a
reference-only Tier-0 (schema + provenance + deck-exists) and skips T1–T4. This notebook
browses them from the manifest. It is fully **PDK-free** and needs no simulator.

Provenance: [`../corpora/ferrosim/PROVENANCE.md`](../corpora/ferrosim/PROVENANCE.md).

## The 22 reference circuits

`catalog.json` is the manifest. Every reference circuit carries a `references` list of
foreign bindings, each indexing its `.scs` decks classified `dut` / `tb` / `runs` / `other`.

In [ ]:
import pandas as pd

from spicexplorer_analog_db import catalog, model, paths

cat = catalog.build_catalog()
refs = [c for c in cat['circuits'] if c.get('kind') == 'reference']


def _deck_paths(c):
    return [p for b in c.get('references', []) for role in ('dut', 'tb', 'runs', 'other')
            for p in b.get(role, [])]


df = pd.DataFrame([{
    'id': c['id'],
    'class': c['class'],
    'nodes': ', '.join(sorted({(b.get('node') or 'va') for b in c['references']})),
    'decks': len(_deck_paths(c)),
} for c in refs]).set_index('id')
print(f"{len(df)} reference circuits | {df['decks'].sum()} decks | "
      f"{df['class'].nunique()} classes")
df

## One reference circuit, up close

`ferrosim_amp5t` is a five-transistor differential-to-single-ended amplifier (a 5T OTA). Its
binding preserves the upstream `dut/` + `tb/` + `runs/` layout verbatim, so a testbench and
its DUT travel together.

In [ ]:
ckt = model.load_circuit('ferrosim_amp5t')
print('kind:', ckt.kind, '| class:', ckt.klass, '| status:', ckt.status)
print('reference-only (skips T1-4):', ckt.is_reference_only)
print('provenance:', {k: ckt.manifest['provenance'][k] for k in ('source', 'designer', 'license')})

entry = next(c for c in refs if c['id'] == 'ferrosim_amp5t')
binding = entry['references'][0]
print('\nbinding:', binding['dir'], '| tool:', binding['tool'], '| node:', binding['node'])
for role in ('dut', 'tb', 'runs'):
    for p in binding.get(role, []):
        print(f'  {role:5s} {p}')

### Read a deck

The decks are plain Spectre text (proprietary-PDK includes stubbed as `${PDK_ROOT}`
placeholders). Catalog paths are db-root-relative — resolve them through `paths.db_root()`.

In [ ]:
dut_rel = binding['dut'][0]
text = (paths.db_root() / dut_rel).read_text()
print(dut_rel, f'({len(text.splitlines())} lines)\n')
print('\n'.join(text.splitlines()[:16]))

## Why they're reference-only

These decks target a proprietary foundry PDK we don't vendor — they can't be lowered to the
open PDKs or simulated here. The verify harness reflects that: a reference circuit passes a
reference-only Tier-0 and **skips** T1–T4.

In [ ]:
from spicexplorer_analog_db import verify

results = verify.run([0, 1, 2], circuit_ids=['ferrosim_amp5t'])
for r in results:
    print(f'  [{r.status:4s}] T{r.tier} {r.check}'
          + (f'  — {r.reason}' if r.reason else ''))
print('\nderived status:', verify.derive_status('ferrosim_amp5t', results))